# QC: можно ли собрать типы карт

Проверка **без витрины и без Excel**. Только Impala.

Канон техлида / `datamart_month_acquiring_v3`: тип карты = описание **FIID эмитента**.

| Что | Откуда |
|---|---|
| Транзакция | `ods_alpha.scd1_trx` |
| Ключ эмитента | `t.c_fiid_iss` |
| Ключ эквайера | `t.c_fiid_acq` (контроль, не тип карты) |
| Справочник | `ods_alpha.scd1_base24_fiids` |
| Тип карты | `f.c_fiid_desc` по `f.c_fiid = t.c_fiid_iss` |
| Группа | `f.c_fiid_grp` |

Периметр как в `raw_trx`: `SA` / `S01` / не `R` / не deleted / есть терминал.

В конце **VERDICT**: собирается / дыра колонок / дыра джойна / в `c_fiid_desc` нет брендов.

Новый kernel нормален. Нужен Impala (`tech.keytab` на `/home/jovyan`).


In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 220)
pd.set_option('display.max_rows', 80)

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
OUT_DIR = DATA_DIR / 'qc_card_types_fiid'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Окно probe: неделя быстрее полного месяца. Для полной картины: 'month'.
PROBE_MODE = 'week'
MONTH_START = '2026-06-01'
MONTH_END = '2026-06-30'
WEEK_END = '2026-06-07'

TRX_TABLE = 'ods_alpha.scd1_trx'
FIID_TABLE = 'ods_alpha.scd1_base24_fiids'
MEM_LIMIT = '8g'
SAMPLE_N = 30
JOIN_OK_PCT = 80.0
BRAND_OK_PCT = 50.0
NOTEBOOK_REV = '2026-09-18-card-types-v1'

impala_user = 'Shestopalov-VYur'
impala_keytab = '/home/jovyan/test_requests/tech.keytab'

probe_start = MONTH_START
probe_end = WEEK_END if PROBE_MODE == 'week' else MONTH_END
print('rev', NOTEBOOK_REV)
print('probe', PROBE_MODE, probe_start, '→', probe_end)
print('OUT_DIR', OUT_DIR)


## 0) Подключение Impala


In [ ]:
def fetch_imp(sql, label, allow_fail=False):
    t0 = pd.Timestamp.now()
    print(f'FETCH {label} ...')
    try:
        with imp:
            imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
            df = imp.fetch(sql)
        if df is None:
            df = pd.DataFrame()
        print(f'  rows={len(df):,}  {(pd.Timestamp.now() - t0).total_seconds():.1f}s')
        return df
    except Exception as exc:
        print(f'  FAIL {type(exc).__name__}: {exc}')
        if allow_fail:
            return None
        raise


def cols_lower(df):
    if df is None or df.empty:
        return []
    return [str(c).strip().lower() for c in df.columns]


def has_col(names, *needles):
    low = {str(x).strip().lower() for x in names}
    return all(n.lower() in low for n in needles)


def classify_brand(text):
    s = '' if pd.isna(text) else str(text).strip().lower()
    if not s or s in {'none', 'nan', 'null', 'unknown'}:
        return 'empty'
    if re.search(r'union\s*pay|unionpay|юнион', s):
        return 'UnionPay'
    if re.search(r'\bmir\b|мир|nspk|нс пк|нспк', s):
        return 'МИР'
    if re.search(r'master\s*card|mastercard|\bmaestro\b|\bmc\b', s):
        return 'Mastercard'
    if re.search(r'\bvisa\b|виза', s):
        return 'Visa'
    if re.search(r'\bjcb\b', s):
        return 'JCB'
    if re.search(r'amex|american\s*express', s):
        return 'Amex'
    if re.search(r'\bsbp\b|сбп', s):
        return 'СБП'
    return 'other'


if 'imp' in globals() and imp is not None:
    print('Reuse existing imp')
else:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': impala_keytab,
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': impala_user},
    )
    imp._init_connection()
    print('Impala connected')

ping = fetch_imp('SELECT current_user() AS u, now() AS ts', 'ping')
display(ping)


## 1) Схемы: есть ли колонки типа карты

Нужны `c_fiid_iss` в `scd1_trx` и `c_fiid` + `c_fiid_desc` в `scd1_base24_fiids`.
Если в справочнике есть ещё поля бренда — покажем их.


In [ ]:
trx_desc = fetch_imp(f'DESCRIBE {TRX_TABLE}', 'DESCRIBE trx', allow_fail=True)
fiid_desc = fetch_imp(f'DESCRIBE {FIID_TABLE}', 'DESCRIBE fiids', allow_fail=True)

if trx_desc is None or fiid_desc is None:
    raise RuntimeError('DESCRIBE не прошёл. Сначала почини Impala / имена таблиц.')

trx_cols = [str(r.iloc[0]).strip().lower() for _, r in trx_desc.iterrows()]
fiid_cols = [str(r.iloc[0]).strip().lower() for _, r in fiid_desc.iterrows()]

need_trx = ['n_trx', 'c_fiid_iss', 'c_fiid_acq', 'd_trx_orig', 'c_trx_class', 'c_trx_type']
need_fiid = ['c_fiid', 'c_fiid_desc']
optional_fiid = [c for c in fiid_cols if re.search(r'brand|card|ps|pay|mir|visa|desc|grp|type|prod', c)]

schema_ok = {
    'trx_ok': has_col(trx_cols, *need_trx),
    'fiid_ok': has_col(fiid_cols, *need_fiid),
    'fiid_has_grp': 'c_fiid_grp' in fiid_cols,
    'missing_trx': [c for c in need_trx if c not in trx_cols],
    'missing_fiid': [c for c in need_fiid if c not in fiid_cols],
}
print('schema:', schema_ok)
print('fiid columns that look brand-related:', optional_fiid or '—')
print('=== trx ===')
display(trx_desc)
print('=== fiids ===')
display(fiid_desc)

if not schema_ok['trx_ok'] or not schema_ok['fiid_ok']:
    raise RuntimeError(
        'Нет обязательных колонок. trx missing='
        + str(schema_ok['missing_trx'])
        + ' fiid missing='
        + str(schema_ok['missing_fiid'])
    )


## 2) Каталог FIID: какие типы вообще есть в справочнике

Если здесь уже есть Visa / Mastercard / МИР / UnionPay — справочник живой, тип карты собрать можно.


In [ ]:
fiid_grp_expr = 'cast(c_fiid_grp as string)' if schema_ok['fiid_has_grp'] else "cast(null as string)"

catalog = fetch_imp(
    f'''
    SELECT
      cast(c_fiid as string) AS c_fiid,
      {fiid_grp_expr} AS c_fiid_grp,
      cast(c_fiid_desc as string) AS c_fiid_desc
    FROM {FIID_TABLE}
    ''',
    'fiid catalog',
)

catalog['c_fiid'] = catalog['c_fiid'].astype(str).str.strip()
catalog['c_fiid_grp'] = catalog['c_fiid_grp'].astype(str).str.strip()
catalog['c_fiid_desc'] = catalog['c_fiid_desc'].astype(str).str.strip()
catalog['brand'] = catalog['c_fiid_desc'].map(classify_brand)

brand_catalog = (
    catalog.groupby('brand', dropna=False)
    .agg(fiid_cnt=('c_fiid', 'nunique'), desc_cnt=('c_fiid_desc', 'nunique'))
    .reset_index()
    .sort_values('fiid_cnt', ascending=False)
)
grp_catalog = (
    catalog.groupby('c_fiid_grp', dropna=False)
    .agg(fiid_cnt=('c_fiid', 'nunique'))
    .reset_index()
    .sort_values('fiid_cnt', ascending=False)
)

print(f'catalog rows={len(catalog):,} unique fiid={catalog["c_fiid"].nunique():,}')
print('=== brand в справочнике ===')
display(brand_catalog)
print('=== c_fiid_grp ===')
display(grp_catalog.head(30))
print('=== примеры c_fiid_desc по бренду ===')
examples = (
    catalog.groupby('brand', as_index=False)
    .agg(sample_desc=('c_fiid_desc', lambda s: ' | '.join(sorted(set(s.astype(str)))[:5])))
)
display(examples)


## 3) Покрытие на живых транзакциях

Считаем в SQL, строки trx на клиент не тянем.

- `iss_joined_pct` — доля покупок с джойном эмитента (это и есть тип карты).
- `iss_vs_acq_diff_pct` — как часто описание эмитента ≠ эквайера. Если низкое и эквайер RSHB — `c_fiid_acq` нельзя брать как тип карты.


In [ ]:
coverage = fetch_imp(
    f'''
    SELECT
      count(*) AS trx_cnt,
      count(t.c_fiid_iss) AS iss_filled,
      count(t.c_fiid_acq) AS acq_filled,
      count(fi.c_fiid) AS iss_joined,
      count(nullif(trim(cast(fi.c_fiid_desc as string)), '')) AS iss_desc_filled,
      count(fa.c_fiid) AS acq_joined,
      count(nullif(trim(cast(fa.c_fiid_desc as string)), '')) AS acq_desc_filled,
      sum(
        CASE
          WHEN coalesce(trim(cast(fi.c_fiid_desc as string)), '')
             <> coalesce(trim(cast(fa.c_fiid_desc as string)), '')
          THEN 1 ELSE 0
        END
      ) AS iss_acq_desc_diff
    FROM {TRX_TABLE} t
    LEFT JOIN {FIID_TABLE} fi
      ON cast(fi.c_fiid as string) = cast(t.c_fiid_iss as string)
    LEFT JOIN {FIID_TABLE} fa
      ON cast(fa.c_fiid as string) = cast(t.c_fiid_acq as string)
    WHERE to_date(cast(t.d_trx_orig as timestamp))
            BETWEEN cast('{probe_start}' as date) AND cast('{probe_end}' as date)
      AND t.c_nter IS NOT NULL
      AND coalesce(t.ods_deleted_flg, '0') <> '1'
      AND t.c_trx_class = 'SA'
      AND t.c_trx_type = 'S01'
      AND coalesce(t.cf_trx_stat, '') <> 'R'
    ''',
    'trx coverage issuer vs acquirer',
)

cov = coverage.iloc[0].to_dict() if len(coverage) else {}
trx_cnt = float(cov.get('trx_cnt') or 0)
for key in list(cov):
    try:
        cov[key] = float(cov[key] or 0)
    except Exception:
        pass

def pct(part, total):
    return 0.0 if not total else 100.0 * float(part) / float(total)

coverage_pct = pd.DataFrame([
    {
        'trx_cnt': int(trx_cnt),
        'iss_filled_pct': round(pct(cov.get('iss_filled'), trx_cnt), 2),
        'iss_joined_pct': round(pct(cov.get('iss_joined'), trx_cnt), 2),
        'iss_desc_pct': round(pct(cov.get('iss_desc_filled'), trx_cnt), 2),
        'acq_joined_pct': round(pct(cov.get('acq_joined'), trx_cnt), 2),
        'acq_desc_pct': round(pct(cov.get('acq_desc_filled'), trx_cnt), 2),
        'iss_vs_acq_diff_pct': round(pct(cov.get('iss_acq_desc_diff'), trx_cnt), 2),
    }
])
print('probe', probe_start, probe_end)
display(coverage)
display(coverage_pct)


## 4) Какие типы карт реально стоят на транзакциях

Агрегат по `c_fiid_iss` → `c_fiid_desc`. Это ответ: «собрать типы карт можно / нельзя».


In [ ]:
live_types = fetch_imp(
    f'''
    SELECT
      cast(fi.c_fiid as string) AS c_fiid_iss,
      cast(fi.c_fiid_grp as string) AS c_fiid_iss_grp,
      cast(fi.c_fiid_desc as string) AS card_type,
      count(*) AS trx_cnt,
      sum(cast(t.n_amt_src as double)) AS trx_sum
    FROM {TRX_TABLE} t
    LEFT JOIN {FIID_TABLE} fi
      ON cast(fi.c_fiid as string) = cast(t.c_fiid_iss as string)
    WHERE to_date(cast(t.d_trx_orig as timestamp))
            BETWEEN cast('{probe_start}' as date) AND cast('{probe_end}' as date)
      AND t.c_nter IS NOT NULL
      AND coalesce(t.ods_deleted_flg, '0') <> '1'
      AND t.c_trx_class = 'SA'
      AND t.c_trx_type = 'S01'
      AND coalesce(t.cf_trx_stat, '') <> 'R'
    GROUP BY 1, 2, 3
    ORDER BY trx_cnt DESC
    ''',
    'live card types on trx',
)

if live_types is None or live_types.empty:
    live_types = pd.DataFrame(columns=['c_fiid_iss', 'c_fiid_iss_grp', 'card_type', 'trx_cnt', 'trx_sum'])

live_types['trx_cnt'] = pd.to_numeric(live_types['trx_cnt'], errors='coerce').fillna(0)
live_types['trx_sum'] = pd.to_numeric(live_types['trx_sum'], errors='coerce').fillna(0)
live_types['brand'] = live_types['card_type'].map(classify_brand)
live_total = float(live_types['trx_cnt'].sum())

brand_live = (
    live_types.groupby('brand', dropna=False)
    .agg(trx_cnt=('trx_cnt', 'sum'), trx_sum=('trx_sum', 'sum'), fiid_cnt=('c_fiid_iss', 'nunique'))
    .reset_index()
)
brand_live['trx_pct'] = np.where(live_total, 100.0 * brand_live['trx_cnt'] / live_total, 0.0)
brand_live = brand_live.sort_values('trx_cnt', ascending=False)

print(f'live types={len(live_types):,} trx={int(live_total):,}')
print('=== бренд на транзакциях ===')
display(brand_live)
print('=== топ card_type ===')
display(live_types.head(40))


## 5) Пример строк: эмитент vs эквайер


In [ ]:
sample = fetch_imp(
    f'''
    SELECT
      cast(t.n_trx as string) AS n_trx,
      cast(to_date(cast(t.d_trx_orig as timestamp)) as string) AS trx_date,
      cast(t.c_nter as string) AS c_nter,
      cast(t.n_mcc as string) AS mcc_code,
      cast(t.c_fiid_iss as string) AS c_fiid_iss,
      cast(fi.c_fiid_grp as string) AS iss_grp,
      cast(fi.c_fiid_desc as string) AS card_type,
      cast(t.c_fiid_acq as string) AS c_fiid_acq,
      cast(fa.c_fiid_grp as string) AS acq_grp,
      cast(fa.c_fiid_desc as string) AS acq_desc,
      cast(t.n_amt_src as double) AS trx_sum
    FROM {TRX_TABLE} t
    LEFT JOIN {FIID_TABLE} fi
      ON cast(fi.c_fiid as string) = cast(t.c_fiid_iss as string)
    LEFT JOIN {FIID_TABLE} fa
      ON cast(fa.c_fiid as string) = cast(t.c_fiid_acq as string)
    WHERE to_date(cast(t.d_trx_orig as timestamp))
            BETWEEN cast('{probe_start}' as date) AND cast('{probe_end}' as date)
      AND t.c_nter IS NOT NULL
      AND coalesce(t.ods_deleted_flg, '0') <> '1'
      AND t.c_trx_class = 'SA'
      AND t.c_trx_type = 'S01'
      AND coalesce(t.cf_trx_stat, '') <> 'R'
    LIMIT {SAMPLE_N}
    ''',
    f'sample {SAMPLE_N} trx',
)

if sample is not None and len(sample):
    sample['brand'] = sample['card_type'].map(classify_brand)
display(sample)


## 6) VERDICT


In [ ]:
known_brands = {'Visa', 'Mastercard', 'МИР', 'UnionPay', 'JCB', 'Amex'}
catalog_known = set(brand_catalog.loc[brand_catalog['brand'].isin(known_brands), 'brand'])
live_known = set(brand_live.loc[brand_live['brand'].isin(known_brands), 'brand'])
live_known_pct = float(brand_live.loc[brand_live['brand'].isin(known_brands), 'trx_pct'].sum()) if len(brand_live) else 0.0
iss_desc_pct = float(coverage_pct.iloc[0]['iss_desc_pct']) if len(coverage_pct) else 0.0
iss_join_pct = float(coverage_pct.iloc[0]['iss_joined_pct']) if len(coverage_pct) else 0.0
diff_pct = float(coverage_pct.iloc[0]['iss_vs_acq_diff_pct']) if len(coverage_pct) else 0.0

if trx_cnt <= 0:
    verdict = 'нет транзакций в окне — смени PROBE_MODE/даты'
elif iss_desc_pct < JOIN_OK_PCT:
    verdict = 'дыра джойна: c_fiid_iss не бьётся со справочником или desc пустой'
elif not catalog_known and not live_known:
    verdict = 'джойн живой, но в c_fiid_desc нет Visa/MC/МИР/UnionPay — это не бренды карт'
elif live_known_pct < BRAND_OK_PCT:
    verdict = (
        f'собирается частично: бренды есть ({", ".join(sorted(live_known))}), '
        f'но покрывают только {live_known_pct:.1f}% trx'
    )
else:
    verdict = (
        f'собирается: тип карты = scd1_base24_fiids.c_fiid_desc по c_fiid_iss. '
        f'бренды {", ".join(sorted(live_known))}, покрытие desc {iss_desc_pct:.1f}%'
    )

print('=== VERDICT ===')
print(verdict)
print(f'iss_joined={iss_join_pct:.1f}%  iss_desc={iss_desc_pct:.1f}%  iss≠acq={diff_pct:.1f}%')
print('catalog brands:', sorted(catalog_known) or '—')
print('live brands:', sorted(live_known) or '—', f'({live_known_pct:.1f}% trx)')
if diff_pct < 20:
    print('WARN: описания эмитента и эквайера почти совпадают — проверь, что смотришь c_fiid_iss, не acq')

out_xlsx = OUT_DIR / f'qc_card_types_{probe_start}_{probe_end}.xlsx'
with pd.ExcelWriter(out_xlsx, engine='openpyxl') as w:
    brand_catalog.to_excel(w, sheet_name='catalog_brand', index=False)
    grp_catalog.to_excel(w, sheet_name='catalog_grp', index=False)
    coverage.to_excel(w, sheet_name='coverage_raw', index=False)
    coverage_pct.to_excel(w, sheet_name='coverage_pct', index=False)
    brand_live.to_excel(w, sheet_name='live_brand', index=False)
    live_types.head(500).to_excel(w, sheet_name='live_types', index=False)
    if sample is not None and len(sample):
        sample.to_excel(w, sheet_name='sample', index=False)
    pd.DataFrame([{
        'verdict': verdict,
        'probe_start': probe_start,
        'probe_end': probe_end,
        'trx_cnt': int(trx_cnt),
        'iss_joined_pct': iss_join_pct,
        'iss_desc_pct': iss_desc_pct,
        'iss_vs_acq_diff_pct': diff_pct,
        'live_known_pct': live_known_pct,
        'catalog_brands': ', '.join(sorted(catalog_known)),
        'live_brands': ', '.join(sorted(live_known)),
        'rev': NOTEBOOK_REV,
    }]).to_excel(w, sheet_name='verdict', index=False)
print('Saved', out_xlsx)
